# GPU probe — measure D12 before committing quota

The first thing in this project that spends GPU quota. It deliberately runs
**one epoch**, not a hundred: the spec's 8–12 week estimate rests on a step
time nobody has measured, and D18 suggests it may be badly pessimistic.
Measuring first costs ~1 quota-hour; guessing wrong costs weeks.

Answers four things that cannot be known from outside a GPU session:

1. are **both** T4s visible and used? (TPSMM wraps in `DataParallel` itself)
2. does `batch_size: 28` fit in 16GB at 128×128, or does it OOM?
3. what is the real **seconds/step** → hours/run → runs/week at 30h quota
4. does the whole path — mount, extract, dataset, model, loss — actually run

Data arrives via `kernel_sources` (D20): Kaggle auto-zips the preprocess
kernel's output, this mounts it, and we extract in place. Nothing transfers
through the workstation.

In [ ]:
# 1. hardware
import subprocess, sys
import torch

n = torch.cuda.device_count()
print("torch  :", torch.__version__, "| cuda:", torch.version.cuda)
print("gpus   :", n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/2**30:.1f} GiB  sm_{p.major}{p.minor}")
assert n == 2, f"expected 2 T4s, got {n} -- accelerator is set wrong, and every" \
               f" timing below would be off by 2x"
# Turing (sm_75): fp16 tensor cores yes, bf16 no, flash-attn no. Matters for D5.
print("\nbf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# 2. extract the prepared data from the mounted kernel output (D20)
import time, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
zips = list(INPUT.rglob("_output_.zip"))
assert zips, f"no kernel output mounted; saw {[p.name for p in INPUT.iterdir()]}"
src = zips[0]
print("mounted:", src, f"({src.stat().st_size/2**30:.2f} GiB)")

DATA = Path("/kaggle/working/data")
t0 = time.time()
with zipfile.ZipFile(src) as z:
    z.extractall(DATA)
print(f"extracted in {(time.time()-t0)/60:.1f} min")

ROOT = DATA / "lsa64_prepared"
print("train clips:", len(list((ROOT / "train").iterdir())), "(paper: 2800)")
print("test clips :", len(list((ROOT / "test").iterdir())), "(paper: 400)")
assert len(list((ROOT / "train").iterdir())) == 2800
assert len(list((ROOT / "test").iterdir())) == 400

In [ ]:
# 3. repo + a one-epoch probe config derived from the real one, so there is a
#    single source of truth for every hyperparameter the paper pins.
import yaml

r = subprocess.run(
    ["git", "clone", "-q", "--branch", "feat/m0-m1-harness", "--depth", "1",
     "https://github.com/SonLamHG/pgmm.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr

REPO = Path("/kaggle/working/repo")
cfg = yaml.safe_load((REPO / "pgmm/config/lsa64-tpsmm.yaml").read_text())
cfg["dataset_params"]["root_dir"] = str(ROOT)
cfg["train_params"]["num_epochs"] = 1        # PROBE ONLY -- the real run is 100
cfg["train_params"]["checkpoint_freq"] = 1

PROBE = REPO / "third_party/tpsmm/config/_probe.yaml"
PROBE.write_text(yaml.safe_dump(cfg, sort_keys=False))

bs = cfg["train_params"]["batch_size"]
steps = 2800 * cfg["train_params"]["num_repeats"] / bs
print(f"batch_size {bs} | num_repeats {cfg['train_params']['num_repeats']}")
print(f"=> {steps:.0f} steps for this one epoch; 100 epochs would be {steps*100:,.0f}")

In [ ]:
# 4. run one real epoch, sampling GPU memory while it goes.
#    Uses TPSMM's own run.py: the probe must exercise the code path the real
#    run uses, not a lookalike.
import threading

peak = [0.0] * torch.cuda.device_count()
stop = threading.Event()

def sample_memory():
    while not stop.is_set():
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=10)
            for i, line in enumerate(out.stdout.strip().splitlines()):
                peak[i] = max(peak[i], float(line))
        except Exception:
            pass
        stop.wait(5)

watcher = threading.Thread(target=sample_memory, daemon=True)
watcher.start()

t0 = time.time()
proc = subprocess.run(
    [sys.executable, "run.py", "--config", "config/_probe.yaml",
     "--device_ids", "0,1", "--log_dir", "/kaggle/working/log"],
    cwd=str(REPO / "third_party/tpsmm"), capture_output=True, text=True,
)
elapsed = time.time() - t0
stop.set(); watcher.join(timeout=10)

print("exit:", proc.returncode)
print("--- stdout tail ---"); print(proc.stdout[-2500:])
print("--- stderr tail ---"); print(proc.stderr[-2500:])

In [ ]:
# 5. D12: turn the measurement into the numbers the plan actually needs.
print("=" * 60)
print("D12 MEASUREMENT")
print("=" * 60)
for i, mb in enumerate(peak):
    print(f"peak GPU{i} memory : {mb/1024:.1f} GiB / 15.0 GiB")

if proc.returncode != 0:
    print("\nRUN FAILED -- no timing to report. Read the stderr above.")
    print("If it is CUDA OOM, batch_size 28 does not fit and must come down;")
    print("keep the effective batch equal via gradient accumulation.")
else:
    s_per_epoch = elapsed
    s_per_step = s_per_epoch / steps
    h_per_run = s_per_step * steps * 100 / 3600
    print(f"\none epoch          : {s_per_epoch/60:.1f} min ({steps:.0f} steps)")
    print(f"seconds/step       : {s_per_step:.3f}")
    print(f"=> hours per 100-epoch run : {h_per_run:.1f} h")
    print(f"=> sessions per run (9h)   : {h_per_run/9:.1f}")
    print(f"=> runs per week (30h quota): {30/h_per_run:.1f}")
    print(f"=> 6-run matrix (R0..R5)   : {6*h_per_run:.0f} h = {6*h_per_run/30:.1f} weeks of quota")
    print("\nNote: epoch 1 includes dataloader warmup, so this over-estimates")
    print("slightly. PGMM runs will be SLOWER: L_align adds a second generator")
    print("pass and the frozen HRNet runs twice per step.")